## 1. Setup and Run Order

This notebook evaluates section 3.1 (`template_id = 39`) from model output CSV files.

Run order:
1. Install dependencies.
2. (Optional) run **Update ground truth** if local CSV files need GT sync.
3. Run Claude block.
4. Run Gemini block.
5. Run GPT-5.2 block.
6. Run final CSV summary and Excel export.

Expected input files in this folder:
- `3_1_with_answers_claude-opus_4_6.csv`
- `3_1_with_answers_gemini.csv`
- `3_1_with_answers_gpt-5_2.csv`

Note: if your inference notebook produced `3_1_with_answers_gpt_5_2.csv`, rename or copy it to `3_1_with_answers_gpt-5_2.csv` for this notebook.


In [ ]:
!pip install pandas requests tqdm openpyxl

### Optional: Update Ground Truth

This block updates `correct_answer` in local `3_1_with_answers_*.csv` files from the database for `template_id = 39`.
Use it only when CSV files are out of sync with DB ground truth.

Important: paths in this block are hardcoded and may need local adjustment.

In [61]:
from pathlib import Path
import shutil
import sqlite3
import pandas as pd

DB_PATH = Path(r"C:\Users\ritaMZ\WebstormProjects\thesis\backend\unified_database.db")

CSV_PATHS = [
    Path(r"C:\Users\ritaMZ\WebstormProjects\thesis\backend\notebooks\dataset\3_1_full_dataset(1).csv"),
    Path(r"C:\Users\ritaMZ\WebstormProjects\thesis\backend\notebooks\3_1_with_answers_claude-opus_4_6.csv"),
    Path(r"C:\Users\ritaMZ\WebstormProjects\thesis\backend\notebooks\3_1_with_answers_gemini.csv"),
    Path(r"C:\Users\ritaMZ\WebstormProjects\thesis\backend\notebooks\3_1_with_answers_gpt-5_2.csv"),
]

TEMPLATE_IDS = {39}

# Possible column names used as the matching key between the database and CSV files
KEY_CANDIDATES = [
    "question_id",
    "generated_question_id",
    "id",
    "question",
    "generated_question",
    "question_text",
]

# Possible answer column names in CSV files
ANSWER_CANDIDATES = ["correct_answer", "CorrectAnswer"]

# Possible template ID column names in CSV files
TEMPLATE_CANDIDATES = ["template_id", "TemplateID"]


def find_col(df, candidates):
    """
    Find the first matching column in a DataFrame, ignoring case.
    Returns the original column name if found, otherwise None.
    """
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None


def read_csv_safely(path):
    """
    Try reading a CSV file using several common encodings.
    Also enables automatic delimiter detection.
    """
    for enc in ("utf-8-sig", "utf-8", "cp1251"):
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=enc)
        except Exception:
            pass
    raise ValueError(f"Failed to read CSV file: {path}")


# Load only rows for template_id 41 and 42 from the SQLite database
with sqlite3.connect(DB_PATH) as conn:
    db = pd.read_sql_query(
        "SELECT * FROM generated_compliance_questions WHERE template_id IN (41, 42)",
        conn
    )

# Detect the key column and the correct answer column in the database table
db_key = find_col(db, KEY_CANDIDATES)
db_answer = find_col(db, ["correct_answer"])

if not db_key or not db_answer:
    raise ValueError(
        f"Required columns were not found in the database. "
        f"key={db_key}, correct_answer={db_answer}, columns={list(db.columns)}"
    )

# Keep only the mapping needed for updates: key -> correct_answer
# Drop rows with missing values and keep the latest record for duplicate keys
db_updates = (
    db[[db_key, db_answer]]
    .dropna(subset=[db_key, db_answer])
    .drop_duplicates(subset=[db_key], keep="last")
    .copy()
)

# Normalize key values to improve matching
db_updates[db_key] = db_updates[db_key].astype(str).str.strip()

# Process each target CSV file
for csv_path in CSV_PATHS:
    df = read_csv_safely(csv_path)

    # Detect relevant columns in the CSV
    csv_key = find_col(df, KEY_CANDIDATES)
    template_col = find_col(df, TEMPLATE_CANDIDATES)
    answer_cols = [c for c in df.columns if c.lower() in {"correct_answer", "correctanswer"}]

    if not csv_key:
        print(f"[SKIP] {csv_path.name}: matching key column was not found")
        continue

    if not answer_cols:
        print(f"[SKIP] {csv_path.name}: correct_answer / CorrectAnswer column was not found")
        continue

    # Work on a copy and normalize key values before merging
    work = df.copy()
    work[csv_key] = work[csv_key].astype(str).str.strip()

    # Merge CSV data with fresh answers from the database
    merged = work.merge(
        db_updates.rename(columns={db_key: csv_key, db_answer: "__new_answer__"}),
        on=csv_key,
        how="left"
    )

    # Update only rows that received a non-empty answer from the database
    mask = merged["__new_answer__"].notna()

    # If the CSV contains template_id, additionally restrict updates to 41 and 42
    if template_col:
        mask &= pd.to_numeric(merged[template_col], errors="coerce").isin(TEMPLATE_IDS)

    updated_count = int(mask.sum())

    if updated_count == 0:
        print(f"[OK] {csv_path.name}: no updates found")
        continue

    # Overwrite all matching answer columns
    for col in answer_cols:
        merged.loc[mask, col] = merged.loc[mask, "__new_answer__"]

    merged = merged.drop(columns="__new_answer__")

    # Create a backup before overwriting the original file
    backup_path = csv_path.with_suffix(csv_path.suffix + ".bak")
    shutil.copy2(csv_path, backup_path)

    # Save the updated CSV
    merged.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"[OK] {csv_path.name}: updated {updated_count} rows, backup -> {backup_path.name}")

[OK] 3_1_full_dataset(1).csv: no updates found
[OK] 3_1_with_answers_claude-opus_4_6.csv: no updates found
[OK] 3_1_with_answers_gemini.csv: no updates found
[OK] 3_1_with_answers_gpt-5_2.csv: no updates found


## 2. Claude Opus 4.6

### 2.1 Template-Level Metrics (Claude)

Loads the Claude CSV and computes accuracy/F1 per `template_id`.

In [62]:
import pandas as pd
import re

#INPUT = "output_with_answers_recovered.csv"
INPUT = "3_1_with_answers_claude-opus_4_6.csv"
#INPUT = "1_1_1_Eval_Claude_Opus_1.csv"

COL_TEMPLATE = "template_id"
COL_GT = "correct_answer"
COL_PRED = "model_answer"

df = pd.read_csv(INPUT)

def normalize_text(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_label(s: str) -> str:
    t = normalize_text(s)
    if t in {"yes", "true"}:
        return "yes"
    if t in {"no", "false"}:
        return "no"
    return t

def contains_whole_word(text: str, word: str) -> bool:
    if not word:
        return False
    pattern = r"(?:^|\s)" + re.escape(word) + r"(?:$|\s)"
    return re.search(pattern, text) is not None

df["gt_norm"] = df[COL_GT].apply(normalize_label)
df["pred_norm_text"] = df[COL_PRED].apply(normalize_text)

def predicted_label_from_text(pred_text: str, gt_label: str) -> str:
    return gt_label if contains_whole_word(pred_text, gt_label) else ""

df["pred_label"] = [
    predicted_label_from_text(p, g)
    for p, g in zip(df["pred_norm_text"], df["gt_norm"])
]

# correctness
df["is_correct"] = df["pred_label"] == df["gt_norm"]

# ---- metrics per each template_id ----
def group_metrics(g: pd.DataFrame) -> pd.Series:
    n = len(g)
    acc = g["is_correct"].mean() if n else 0.0

    labels = sorted(set(g["gt_norm"].unique()) | set(g["pred_label"].unique()))
    labels = [lab for lab in labels if lab != ""]  # уберём пустой

    # F1:
    # - If only yes/no in GT -> binary F1 (positive=yes)
    # - Otherwise, macro F1 by GT classes (ignoring empty class)
    gt_set = set(g["gt_norm"].unique())
    if gt_set.issubset({"yes", "no"}):
        y_true = g["gt_norm"].tolist()
        y_pred = [("yes" if x == "yes" else "no") if x in {"yes","no"} else "no" for x in g["pred_label"].tolist()]
        # F1 positive=yes
        tp = sum((t == "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fp = sum((t != "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fn = sum((t == "yes") and (p != "yes") for t, p in zip(y_true, y_pred))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        f1_type = "binary(yes)"
    else:
        # macro F1 per GT classes
        y_true = g["gt_norm"].tolist()
        y_pred = g["pred_label"].tolist()

        f1s = []
        for lab in sorted(gt_set):
            tp = sum((t == lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fp = sum((t != lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fn = sum((t == lab) and (p != lab) for t, p in zip(y_true, y_pred))
            precision = tp / (tp + fp) if (tp + fp) else 0.0
            recall = tp / (tp + fn) if (tp + fn) else 0.0
            f1_lab = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
            f1s.append(f1_lab)
        f1 = sum(f1s) / len(f1s) if f1s else 0.0
        f1_type = "macro(GT-classes)"

    return pd.Series({
        "n": n,
        "accuracy": acc,
        "f1": f1,
        "f1_type": f1_type,
    })

rows = []
for template_id, g in df.groupby(COL_TEMPLATE, dropna=False):
    m = group_metrics(g)          # Series: n, accuracy, f1, f1_type
    m[COL_TEMPLATE] = template_id
    rows.append(m)

report = pd.DataFrame(rows).sort_values(COL_TEMPLATE).reset_index(drop=True)
report


,n,accuracy,f1,f1_type,template_id
0,163,0.619632,0.615385,binary(yes),39


### 2.2 Ambiguity and Overall Metrics (Claude)

Builds summary blocks for `Ambiguity=yes`, `Ambiguity=no`, and `Overall`.

In [63]:
import numpy as np
from sklearn.metrics import f1_score

def summarize_subset(df_in: pd.DataFrame, label: str):
    sub = df_in.copy()
    n = len(sub)
    acc = sub["is_correct"].mean() if n else 0.0
    f1 = group_metrics(sub)["f1"] if n else 0.0

    # Label distribution (by ground-truth)
    dist = sub["gt_norm"].value_counts(dropna=False)
    dist_str = ", ".join([f"{k}:{v}" for k, v in dist.items()]) if n else ""

    # Majority baseline (by ground-truth)
    majority_count = int(dist.max()) if n else 0
    majority_acc = (majority_count / n) if n else 0.0


    # F1 breakdown for binary yes/no (treat non-yes as no)
    if n:
        y_true = (sub["gt_norm"] == "yes").astype(int).to_numpy()
        y_pred = (sub["pred_label"] == "yes").astype(int).to_numpy()
        f1_macro = f1_score(y_true, y_pred, average="macro")
        f1_yes = f1_score(y_true, y_pred, pos_label=1)
        f1_no = f1_score(y_true, y_pred, pos_label=0)

        maj_label = sub["gt_norm"].value_counts().idxmax()
        y_base = np.ones_like(y_true) if maj_label == "yes" else np.zeros_like(y_true)
        f1_macro_base = f1_score(y_true, y_base, average="macro")
        f1_yes_base = f1_score(y_true, y_base, pos_label=1)
        f1_no_base = f1_score(y_true, y_base, pos_label=0)
    else:
        f1_macro = f1_yes = f1_no = 0.0
        f1_macro_base = f1_yes_base = f1_no_base = 0.0

    print(
        f"{label}: n={n}, accuracy={acc:.4f}, f1={f1:.4f}, "
        f"f1_macro={f1_macro:.4f}, f1_yes={f1_yes:.4f}, f1_no={f1_no:.4f}, "
        f"majority_baseline={majority_acc:.4f}"
    )
    if n:
        print(f"  label_distribution: {dist_str}")
        print(
            f"  baseline_f1: macro={f1_macro_base:.4f}, yes={f1_yes_base:.4f}, no={f1_no_base:.4f}"
        )

import numpy as np

def normalize_ambiguity(value) -> str:
    s = "" if pd.isna(value) else str(value)
    s = s.strip().lower()
    return "yes" if s in {"yes", "y", "true", "1"} else "no"

def is_filled(val) -> bool:
    return (not pd.isna(val)) and (str(val).strip() != "")

amb_col = next((c for c in df.columns if str(c).strip().lower() == "ambiguity"), None)
if amb_col is not None:
    df["_amb_norm"] = df[amb_col].apply(normalize_ambiguity)
else:
    df["_amb_norm"] = "no"

summarize_subset(df[df["_amb_norm"] == "yes"], "Ambiguity=yes")
summarize_subset(df[df["_amb_norm"] == "no"], "Ambiguity=no")

# Overall
summarize_subset(df, "Overall")


Ambiguity=yes: n=81, accuracy=0.4568, f1=0.3860, f1_macro=0.5263, f1_yes=0.3860, f1_no=0.6667, majority_baseline=0.5679
  label_distribution: yes:46, no:35
  baseline_f1: macro=0.3622, yes=0.7244, no=0.0000
Ambiguity=no: n=82, accuracy=0.7805, f1=0.7945, f1_macro=0.8148, f1_yes=0.7945, f1_no=0.8352, majority_baseline=0.5366
  label_distribution: yes:44, no:38
  baseline_f1: macro=0.3492, yes=0.6984, no=0.0000
Overall: n=163, accuracy=0.6196, f1=0.6154, f1_macro=0.6801, f1_yes=0.6154, f1_no=0.7449, majority_baseline=0.5521
  label_distribution: yes:90, no:73
  baseline_f1: macro=0.3557, yes=0.7115, no=0.0000


### 2.3 Save Claude Metrics Dictionary

Stores Claude metrics in a dictionary used by the final aggregation table.

In [64]:
import numpy as np
import pandas as pd
import re
from sklearn.metrics import f1_score

# Build Claude metrics dictionary for final aggregation

def _normalize_ambiguity(value) -> str:
    s = "" if pd.isna(value) else str(value)
    s = s.strip().lower()
    return "yes" if s in {"yes", "y", "true", "1"} else "no"

def _is_filled(val) -> bool:
    return (not pd.isna(val)) and (str(val).strip() != "")

def _ensure_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    df_local = df_in.copy()
    if "bool_model_answer" not in df_local.columns:
        matches = df_local["model_answer"].fillna("").astype(str).str.findall(r"@\s*(YES|NO)\b", flags=re.IGNORECASE)
        df_local["bool_model_answer"] = matches.apply(lambda m: m[-1].lower() if m else "")
    if "_rule_type" not in df_local.columns:
        ruleid_filled = df_local["RuleID"].apply(_is_filled) if "RuleID" in df_local.columns else pd.Series([False] * len(df_local))
        df_local["_rule_type"] = np.where(ruleid_filled, "Atomic Rule", "Parent Rule")
    if "_amb_norm" not in df_local.columns:
        amb_col = next((c for c in df_local.columns if str(c).strip().lower() == "ambiguity"), None)
        if amb_col is not None:
            df_local["_amb_norm"] = df_local[amb_col].apply(_normalize_ambiguity)
        else:
            df_local["_amb_norm"] = "no"
        df_local.loc[df_local["_rule_type"] == "Parent Rule", "_amb_norm"] = ""
    return df_local

def _compute_metrics(sub: pd.DataFrame) -> dict:
    n = len(sub)
    if n == 0:
        return {
            "n": 0,
            "accuracy": 0.0,
            "f1": 0.0,
            "f1_macro": 0.0,
            "f1_yes": 0.0,
            "f1_no": 0.0,
            "majority_baseline": 0.0,
            "validity": 0.0,
            "label_distribution": "",
            "baseline_f1_macro": 0.0,
            "baseline_f1_yes": 0.0,
            "baseline_f1_no": 0.0,
        }

    acc = sub["is_correct"].mean()
    dist = sub["gt_norm"].value_counts(dropna=False)
    dist_str = ", ".join([f"{k}:{v}" for k, v in dist.items()])
    majority_acc = int(dist.max()) / n

    pred_valid = sub["bool_model_answer"].fillna("").astype(str).str.strip().str.lower()
    if sub["gt_norm"].isin(["yes", "no"]).all():
        valid = pred_valid.isin(["yes", "no"])
    else:
        scope = set(sub["gt_norm"].dropna().unique())
        valid = pred_valid.isin(scope) if scope else (pred_valid != "")
    validity = valid.mean()

    y_true = (sub["gt_norm"] == "yes").astype(int).to_numpy()
    y_pred = (sub["pred_label"] == "yes").astype(int).to_numpy()
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_yes = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_no = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1 = f1_yes

    maj_label = dist.idxmax()
    y_base = np.ones_like(y_true) if maj_label == "yes" else np.zeros_like(y_true)
    f1_macro_base = f1_score(y_true, y_base, average="macro", zero_division=0)
    f1_yes_base = f1_score(y_true, y_base, pos_label=1, zero_division=0)
    f1_no_base = f1_score(y_true, y_base, pos_label=0, zero_division=0)

    return {
        "n": n,
        "accuracy": acc,
        "f1": f1,
        "f1_macro": f1_macro,
        "f1_yes": f1_yes,
        "f1_no": f1_no,
        "majority_baseline": majority_acc,
        "validity": validity,
        "label_distribution": dist_str,
        "baseline_f1_macro": f1_macro_base,
        "baseline_f1_yes": f1_yes_base,
        "baseline_f1_no": f1_no_base,
    }

df = _ensure_columns(df)

CLAUDE_METRICS = {
    "Ambiguity=yes": _compute_metrics(df[df["_amb_norm"] == "yes"]),
    "Ambiguity=no": _compute_metrics(df[df["_amb_norm"] == "no"]),
    "RuleType=Parent Rule": _compute_metrics(df[df["_rule_type"] == "Parent Rule"]),
    "RuleType=Atomic Rule": _compute_metrics(df[df["_rule_type"] == "Atomic Rule"]),
    "Overall": _compute_metrics(df),
}

# Optional unified dictionary for the final table
if "MODEL_METRICS" not in globals() or not isinstance(MODEL_METRICS, dict):
    MODEL_METRICS = {}
MODEL_METRICS["Claude Opus"] = CLAUDE_METRICS

CLAUDE_METRICS


{'Ambiguity=yes': {'n': 81,
  'accuracy': np.float64(0.4567901234567901),
  'f1': 0.38596491228070173,
  'f1_macro': 0.5263157894736842,
  'f1_yes': 0.38596491228070173,
  'f1_no': 0.6666666666666666,
  'majority_baseline': 0.5679012345679012,
  'validity': np.float64(1.0),
  'label_distribution': 'yes:46, no:35',
  'baseline_f1_macro': 0.36220472440944884,
  'baseline_f1_yes': 0.7244094488188977,
  'baseline_f1_no': 0.0},
 'Ambiguity=no': {'n': 82,
  'accuracy': np.float64(0.7804878048780488),
  'f1': 0.7945205479452054,
  'f1_macro': 0.8148426915550203,
  'f1_yes': 0.7945205479452054,
  'f1_no': 0.8351648351648352,
  'majority_baseline': 0.5365853658536586,
  'validity': np.float64(1.0),
  'label_distribution': 'yes:44, no:38',
  'baseline_f1_macro': 0.3492063492063492,
  'baseline_f1_yes': 0.6984126984126984,
  'baseline_f1_no': 0.0},
 'RuleType=Parent Rule': {'n': 163,
  'accuracy': np.float64(0.6196319018404908),
  'f1': 0.6153846153846154,
  'f1_macro': 0.6801412872841445,
  'f1_

## 3. Gemini 3 Pro Preview

### 3.1 Template-Level Metrics (Gemini)

Loads the Gemini CSV and computes accuracy/F1 per `template_id`.

In [65]:
import pandas as pd
import re

#INPUT = "output_with_answers_recovered.csv"
INPUT = "3_1_with_answers_gemini.csv"
#INPUT = "1_1_1_Eval_Claude_Opus_1.csv"

COL_TEMPLATE = "template_id"
COL_GT = "correct_answer"
COL_PRED = "model_answer"

df = pd.read_csv(INPUT)

def normalize_text(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_label(s: str) -> str:
    t = normalize_text(s)
    if t in {"yes", "true"}:
        return "yes"
    if t in {"no", "false"}:
        return "no"
    return t

def contains_whole_word(text: str, word: str) -> bool:
    if not word:
        return False
    pattern = r"(?:^|\s)" + re.escape(word) + r"(?:$|\s)"
    return re.search(pattern, text) is not None

df["gt_norm"] = df[COL_GT].apply(normalize_label)
df["pred_norm_text"] = df[COL_PRED].apply(normalize_text)

def predicted_label_from_text(pred_text: str, gt_label: str) -> str:
    return gt_label if contains_whole_word(pred_text, gt_label) else ""

df["pred_label"] = [
    predicted_label_from_text(p, g)
    for p, g in zip(df["pred_norm_text"], df["gt_norm"])
]

# correctness
df["is_correct"] = df["pred_label"] == df["gt_norm"]

# ---- metrics per each template_id ----
def group_metrics(g: pd.DataFrame) -> pd.Series:
    n = len(g)
    acc = g["is_correct"].mean() if n else 0.0

    labels = sorted(set(g["gt_norm"].unique()) | set(g["pred_label"].unique()))
    labels = [lab for lab in labels if lab != ""]  # уберём пустой

    # F1:
    # - If only yes/no in GT -> binary F1 (positive=yes)
    # - Otherwise, macro F1 by GT classes (ignoring empty class)
    gt_set = set(g["gt_norm"].unique())
    if gt_set.issubset({"yes", "no"}):
        y_true = g["gt_norm"].tolist()
        y_pred = [("yes" if x == "yes" else "no") if x in {"yes","no"} else "no" for x in g["pred_label"].tolist()]
        # F1 positive=yes
        tp = sum((t == "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fp = sum((t != "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fn = sum((t == "yes") and (p != "yes") for t, p in zip(y_true, y_pred))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        f1_type = "binary(yes)"
    else:
        # macro F1 per GT classes
        y_true = g["gt_norm"].tolist()
        y_pred = g["pred_label"].tolist()

        f1s = []
        for lab in sorted(gt_set):
            tp = sum((t == lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fp = sum((t != lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fn = sum((t == lab) and (p != lab) for t, p in zip(y_true, y_pred))
            precision = tp / (tp + fp) if (tp + fp) else 0.0
            recall = tp / (tp + fn) if (tp + fn) else 0.0
            f1_lab = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
            f1s.append(f1_lab)
        f1 = sum(f1s) / len(f1s) if f1s else 0.0
        f1_type = "macro(GT-classes)"

    return pd.Series({
        "n": n,
        "accuracy": acc,
        "f1": f1,
        "f1_type": f1_type,
    })

rows = []
for template_id, g in df.groupby(COL_TEMPLATE, dropna=False):
    m = group_metrics(g)          # Series: n, accuracy, f1, f1_type
    m[COL_TEMPLATE] = template_id
    rows.append(m)

report = pd.DataFrame(rows).sort_values(COL_TEMPLATE).reset_index(drop=True)
report


,n,accuracy,f1,f1_type,template_id
0,163,0.595092,0.56,binary(yes),39


### 3.2 Ambiguity and Overall Metrics (Gemini)

Builds summary blocks for `Ambiguity=yes`, `Ambiguity=no`, and `Overall`.

In [66]:
import numpy as np
from sklearn.metrics import f1_score

def summarize_subset(df_in: pd.DataFrame, label: str):
    sub = df_in.copy()
    n = len(sub)
    acc = sub["is_correct"].mean() if n else 0.0
    f1 = group_metrics(sub)["f1"] if n else 0.0

    # Label distribution (by ground-truth)
    dist = sub["gt_norm"].value_counts(dropna=False)
    dist_str = ", ".join([f"{k}:{v}" for k, v in dist.items()]) if n else ""

    # Majority baseline (by ground-truth)
    majority_count = int(dist.max()) if n else 0
    majority_acc = (majority_count / n) if n else 0.0


    # F1 breakdown for binary yes/no (treat non-yes as no)
    if n:
        y_true = (sub["gt_norm"] == "yes").astype(int).to_numpy()
        y_pred = (sub["pred_label"] == "yes").astype(int).to_numpy()
        f1_macro = f1_score(y_true, y_pred, average="macro")
        f1_yes = f1_score(y_true, y_pred, pos_label=1)
        f1_no = f1_score(y_true, y_pred, pos_label=0)

        maj_label = sub["gt_norm"].value_counts().idxmax()
        y_base = np.ones_like(y_true) if maj_label == "yes" else np.zeros_like(y_true)
        f1_macro_base = f1_score(y_true, y_base, average="macro")
        f1_yes_base = f1_score(y_true, y_base, pos_label=1)
        f1_no_base = f1_score(y_true, y_base, pos_label=0)
    else:
        f1_macro = f1_yes = f1_no = 0.0
        f1_macro_base = f1_yes_base = f1_no_base = 0.0

    print(
        f"{label}: n={n}, accuracy={acc:.4f}, f1={f1:.4f}, "
        f"f1_macro={f1_macro:.4f}, f1_yes={f1_yes:.4f}, f1_no={f1_no:.4f}, "
        f"majority_baseline={majority_acc:.4f}"
    )
    if n:
        print(f"  label_distribution: {dist_str}")
        print(
            f"  baseline_f1: macro={f1_macro_base:.4f}, yes={f1_yes_base:.4f}, no={f1_no_base:.4f}"
        )

import numpy as np

def normalize_ambiguity(value) -> str:
    s = "" if pd.isna(value) else str(value)
    s = s.strip().lower()
    return "yes" if s in {"yes", "y", "true", "1"} else "no"

amb_col = next((c for c in df.columns if str(c).strip().lower() == "ambiguity"), None)
if amb_col is not None:
    df["_amb_norm"] = df[amb_col].apply(normalize_ambiguity)
else:
    df["_amb_norm"] = "no"

### 3.3 Save Gemini Metrics Dictionary

Stores Gemini metrics in a dictionary used by the final aggregation table.

In [67]:
import numpy as np
import pandas as pd
import re
from sklearn.metrics import f1_score

# Build Gemini metrics dictionary for final aggregation

def _normalize_ambiguity(value) -> str:
    s = "" if pd.isna(value) else str(value)
    s = s.strip().lower()
    return "yes" if s in {"yes", "y", "true", "1"} else "no"

def _is_filled(val) -> bool:
    return (not pd.isna(val)) and (str(val).strip() != "")

def _ensure_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    df_local = df_in.copy()
    if "bool_model_answer" not in df_local.columns:
        matches = df_local["model_answer"].fillna("").astype(str).str.findall(r"@\s*(YES|NO)\b", flags=re.IGNORECASE)
        df_local["bool_model_answer"] = matches.apply(lambda m: m[-1].lower() if m else "")
    if "_rule_type" not in df_local.columns:
        ruleid_filled = df_local["RuleID"].apply(_is_filled) if "RuleID" in df_local.columns else pd.Series([False] * len(df_local))
        df_local["_rule_type"] = np.where(ruleid_filled, "Atomic Rule", "Parent Rule")
    if "_amb_norm" not in df_local.columns:
        amb_col = next((c for c in df_local.columns if str(c).strip().lower() == "ambiguity"), None)
        if amb_col is not None:
            df_local["_amb_norm"] = df_local[amb_col].apply(_normalize_ambiguity)
        else:
            df_local["_amb_norm"] = "no"
        df_local.loc[df_local["_rule_type"] == "Parent Rule", "_amb_norm"] = ""
    return df_local

def _compute_metrics(sub: pd.DataFrame) -> dict:
    n = len(sub)
    if n == 0:
        return {
            "n": 0,
            "accuracy": 0.0,
            "f1": 0.0,
            "f1_macro": 0.0,
            "f1_yes": 0.0,
            "f1_no": 0.0,
            "majority_baseline": 0.0,
            "validity": 0.0,
            "label_distribution": "",
            "baseline_f1_macro": 0.0,
            "baseline_f1_yes": 0.0,
            "baseline_f1_no": 0.0,
        }

    acc = sub["is_correct"].mean()
    dist = sub["gt_norm"].value_counts(dropna=False)
    dist_str = ", ".join([f"{k}:{v}" for k, v in dist.items()])
    majority_acc = int(dist.max()) / n

    pred_valid = sub["bool_model_answer"].fillna("").astype(str).str.strip().str.lower()
    if sub["gt_norm"].isin(["yes", "no"]).all():
        valid = pred_valid.isin(["yes", "no"])
    else:
        scope = set(sub["gt_norm"].dropna().unique())
        valid = pred_valid.isin(scope) if scope else (pred_valid != "")
    validity = valid.mean()

    y_true = (sub["gt_norm"] == "yes").astype(int).to_numpy()
    y_pred = (sub["pred_label"] == "yes").astype(int).to_numpy()
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_yes = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_no = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1 = f1_yes

    maj_label = dist.idxmax()
    y_base = np.ones_like(y_true) if maj_label == "yes" else np.zeros_like(y_true)
    f1_macro_base = f1_score(y_true, y_base, average="macro", zero_division=0)
    f1_yes_base = f1_score(y_true, y_base, pos_label=1, zero_division=0)
    f1_no_base = f1_score(y_true, y_base, pos_label=0, zero_division=0)

    return {
        "n": n,
        "accuracy": acc,
        "f1": f1,
        "f1_macro": f1_macro,
        "f1_yes": f1_yes,
        "f1_no": f1_no,
        "majority_baseline": majority_acc,
        "validity": validity,
        "label_distribution": dist_str,
        "baseline_f1_macro": f1_macro_base,
        "baseline_f1_yes": f1_yes_base,
        "baseline_f1_no": f1_no_base,
    }

df = _ensure_columns(df)

GEMINI_METRICS = {
    "Ambiguity=yes": _compute_metrics(df[df["_amb_norm"] == "yes"]),
    "Ambiguity=no": _compute_metrics(df[df["_amb_norm"] == "no"]),
    "RuleType=Parent Rule": _compute_metrics(df[df["_rule_type"] == "Parent Rule"]),
    "RuleType=Atomic Rule": _compute_metrics(df[df["_rule_type"] == "Atomic Rule"]),
    "Overall": _compute_metrics(df),
}

# Optional unified dictionary for the final table
if "MODEL_METRICS" not in globals() or not isinstance(MODEL_METRICS, dict):
    MODEL_METRICS = {}
MODEL_METRICS["Gemini 3 Pro Preview"] = GEMINI_METRICS

GEMINI_METRICS


{'Ambiguity=yes': {'n': 81,
  'accuracy': np.float64(0.43209876543209874),
  'f1': 0.2641509433962264,
  'f1_macro': 0.4531763891293059,
  'f1_yes': 0.2641509433962264,
  'f1_no': 0.6422018348623854,
  'majority_baseline': 0.5679012345679012,
  'validity': np.float64(0.9876543209876543),
  'label_distribution': 'yes:46, no:35',
  'baseline_f1_macro': 0.36220472440944884,
  'baseline_f1_yes': 0.7244094488188977,
  'baseline_f1_no': 0.0},
 'Ambiguity=no': {'n': 82,
  'accuracy': np.float64(0.7560975609756098),
  'f1': 0.7777777777777778,
  'f1_macro': 0.8019323671497585,
  'f1_yes': 0.7777777777777778,
  'f1_no': 0.8260869565217391,
  'majority_baseline': 0.5365853658536586,
  'validity': np.float64(1.0),
  'label_distribution': 'yes:44, no:38',
  'baseline_f1_macro': 0.3492063492063492,
  'baseline_f1_yes': 0.6984126984126984,
  'baseline_f1_no': 0.0},
 'RuleType=Parent Rule': {'n': 163,
  'accuracy': np.float64(0.5950920245398773),
  'f1': 0.56,
  'f1_macro': 0.6431840796019901,
  'f1_

## 4. GPT-5.2

### 4.1 Template-Level Metrics (GPT-5.2)

Loads the GPT CSV and computes accuracy/F1 per `template_id`.

In [68]:
import pandas as pd
import re

INPUT = "3_1_with_answers_gpt-5_2.csv"


COL_TEMPLATE = "template_id"
COL_GT = "correct_answer"
COL_PRED = "model_answer"

df = pd.read_csv(INPUT)

def normalize_text(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_label(s: str) -> str:
    t = normalize_text(s)
    if t in {"yes", "true"}:
        return "yes"
    if t in {"no", "false"}:
        return "no"
    return t

def contains_whole_word(text: str, word: str) -> bool:
    if not word:
        return False
    pattern = r"(?:^|\s)" + re.escape(word) + r"(?:$|\s)"
    return re.search(pattern, text) is not None

df["gt_norm"] = df[COL_GT].apply(normalize_label)
df["pred_norm_text"] = df[COL_PRED].apply(normalize_text)

def predicted_label_from_text(pred_text: str, gt_label: str) -> str:
    return gt_label if contains_whole_word(pred_text, gt_label) else ""

df["pred_label"] = [
    predicted_label_from_text(p, g)
    for p, g in zip(df["pred_norm_text"], df["gt_norm"])
]

# correctness
df["is_correct"] = df["pred_label"] == df["gt_norm"]

# ---- metrics per each template_id ----
def group_metrics(g: pd.DataFrame) -> pd.Series:
    n = len(g)
    acc = g["is_correct"].mean() if n else 0.0

    labels = sorted(set(g["gt_norm"].unique()) | set(g["pred_label"].unique()))
    labels = [lab for lab in labels if lab != ""]  # уберём пустой

    # F1:
    # - If only yes/no in GT -> binary F1 (positive=yes)
    # - Otherwise, macro F1 by GT classes (ignoring empty class)
    gt_set = set(g["gt_norm"].unique())
    if gt_set.issubset({"yes", "no"}):
        y_true = g["gt_norm"].tolist()
        y_pred = [("yes" if x == "yes" else "no") if x in {"yes","no"} else "no" for x in g["pred_label"].tolist()]
        # F1 positive=yes
        tp = sum((t == "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fp = sum((t != "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fn = sum((t == "yes") and (p != "yes") for t, p in zip(y_true, y_pred))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        f1_type = "binary(yes)"
    else:
        # macro F1 per GT classes
        y_true = g["gt_norm"].tolist()
        y_pred = g["pred_label"].tolist()

        f1s = []
        for lab in sorted(gt_set):
            tp = sum((t == lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fp = sum((t != lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fn = sum((t == lab) and (p != lab) for t, p in zip(y_true, y_pred))
            precision = tp / (tp + fp) if (tp + fp) else 0.0
            recall = tp / (tp + fn) if (tp + fn) else 0.0
            f1_lab = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
            f1s.append(f1_lab)
        f1 = sum(f1s) / len(f1s) if f1s else 0.0
        f1_type = "macro(GT-classes)"

    return pd.Series({
        "n": n,
        "accuracy": acc,
        "f1": f1,
        "f1_type": f1_type,
    })

rows = []
for template_id, g in df.groupby(COL_TEMPLATE, dropna=False):
    m = group_metrics(g)          # Series: n, accuracy, f1, f1_type
    m[COL_TEMPLATE] = template_id
    rows.append(m)

report = pd.DataFrame(rows).sort_values(COL_TEMPLATE).reset_index(drop=True)
report


,n,accuracy,f1,f1_type,template_id
0,163,0.478528,0.2,binary(yes),39


### 4.2 Ambiguity, Rule Type, and Overall Metrics (GPT-5.2)

Builds summary blocks for ambiguity split, parent-vs-atomic split, and overall metrics.

In [69]:
import numpy as np
from sklearn.metrics import f1_score

def summarize_subset(df_in: pd.DataFrame, label: str):
    sub = df_in.copy()
    n = len(sub)
    acc = sub["is_correct"].mean() if n else 0.0
    f1 = group_metrics(sub)["f1"] if n else 0.0

    # Label distribution (by ground-truth)
    dist = sub["gt_norm"].value_counts(dropna=False)
    dist_str = ", ".join([f"{k}:{v}" for k, v in dist.items()]) if n else ""

    # Majority baseline (by ground-truth)
    majority_count = int(dist.max()) if n else 0
    majority_acc = (majority_count / n) if n else 0.0

    # Validity: prediction is in the answer scope for the question type
    if n:
        pred_valid = sub["bool_model_answer"].fillna("").astype(str).str.strip().str.lower()
        if sub["gt_norm"].isin(["yes", "no"]).all():
            # binary scope
            valid = pred_valid.isin(["yes", "no"])
        else:
            # fallback scope: use observed GT labels in this subset
            scope = set(sub["gt_norm"].dropna().unique())
            valid = pred_valid.isin(scope) if scope else (pred_valid != "")
    else:
        valid = pd.Series([], dtype=bool)
    validity = valid.mean() if n else 0.0

    # F1 breakdown for binary yes/no (treat non-yes as no)
    if n:
        y_true = (sub["gt_norm"] == "yes").astype(int).to_numpy()
        y_pred = (sub["pred_label"] == "yes").astype(int).to_numpy()
        f1_macro = f1_score(y_true, y_pred, average="macro")
        f1_yes = f1_score(y_true, y_pred, pos_label=1)
        f1_no = f1_score(y_true, y_pred, pos_label=0)

        maj_label = sub["gt_norm"].value_counts().idxmax()
        y_base = np.ones_like(y_true) if maj_label == "yes" else np.zeros_like(y_true)
        f1_macro_base = f1_score(y_true, y_base, average="macro")
        f1_yes_base = f1_score(y_true, y_base, pos_label=1)
        f1_no_base = f1_score(y_true, y_base, pos_label=0)
    else:
        f1_macro = f1_yes = f1_no = 0.0
        f1_macro_base = f1_yes_base = f1_no_base = 0.0

    print(
        f"{label}: n={n}, accuracy={acc:.4f}, f1={f1:.4f}, "
        f"f1_macro={f1_macro:.4f}, f1_yes={f1_yes:.4f}, f1_no={f1_no:.4f}, "
        f"majority_baseline={majority_acc:.4f}"
    )
    if n:
        print(f"  label_distribution: {dist_str}")
        print(
            f"  baseline_f1: macro={f1_macro_base:.4f}, yes={f1_yes_base:.4f}, no={f1_no_base:.4f}"
        )

import numpy as np

def normalize_ambiguity(value) -> str:
    s = "" if pd.isna(value) else str(value)
    s = s.strip().lower()
    return "yes" if s in {"yes", "y", "true", "1"} else "no"


amb_col = next((c for c in df.columns if str(c).strip().lower() == "ambiguity"), None)
if amb_col is not None:
    df["_amb_norm"] = df[amb_col].apply(normalize_ambiguity)
else:
    df["_amb_norm"] = "no"

summarize_subset(df[df["_amb_norm"] == "yes"], "Ambiguity=yes")
summarize_subset(df[df["_amb_norm"] == "no"], "Ambiguity=no")

# Overall
summarize_subset(df, "Overall")

Ambiguity=yes: n=81, accuracy=0.4198, f1=0.1224, f1_macro=0.3710, f1_yes=0.1224, f1_no=0.6195, majority_baseline=0.5679
  label_distribution: yes:46, no:35
  baseline_f1: macro=0.3622, yes=0.7244, no=0.0000
Ambiguity=no: n=82, accuracy=0.5366, f1=0.2745, f1_macro=0.4735, f1_yes=0.2745, f1_no=0.6726, majority_baseline=0.5366
  label_distribution: yes:44, no:38
  baseline_f1: macro=0.3492, yes=0.6984, no=0.0000
Overall: n=163, accuracy=0.4785, f1=0.2000, f1_macro=0.4230, f1_yes=0.2000, f1_no=0.6460, majority_baseline=0.5521
  label_distribution: yes:90, no:73
  baseline_f1: macro=0.3557, yes=0.7115, no=0.0000


### 4.3 Save GPT-5.2 Metrics Dictionary

Stores GPT-5.2 metrics in a dictionary used by the final aggregation table.

In [70]:
import numpy as np
import pandas as pd
import re
from sklearn.metrics import f1_score

# Build GPT metrics dictionary for final aggregation

def _normalize_ambiguity(value) -> str:
    s = "" if pd.isna(value) else str(value)
    s = s.strip().lower()
    return "yes" if s in {"yes", "y", "true", "1"} else "no"

def _is_filled(val) -> bool:
    return (not pd.isna(val)) and (str(val).strip() != "")

def _ensure_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    df_local = df_in.copy()
    if "bool_model_answer" not in df_local.columns:
        matches = df_local["model_answer"].fillna("").astype(str).str.findall(r"@\s*(YES|NO)\b", flags=re.IGNORECASE)
        df_local["bool_model_answer"] = matches.apply(lambda m: m[-1].lower() if m else "")
    if "_rule_type" not in df_local.columns:
        ruleid_filled = df_local["RuleID"].apply(_is_filled) if "RuleID" in df_local.columns else pd.Series([False] * len(df_local))
        df_local["_rule_type"] = np.where(ruleid_filled, "Atomic Rule", "Parent Rule")
    if "_amb_norm" not in df_local.columns:
        amb_col = next((c for c in df_local.columns if str(c).strip().lower() == "ambiguity"), None)
        if amb_col is not None:
            df_local["_amb_norm"] = df_local[amb_col].apply(_normalize_ambiguity)
        else:
            df_local["_amb_norm"] = "no"
        df_local.loc[df_local["_rule_type"] == "Parent Rule", "_amb_norm"] = ""
    return df_local

def _compute_metrics(sub: pd.DataFrame) -> dict:
    n = len(sub)
    if n == 0:
        return {
            "n": 0,
            "accuracy": 0.0,
            "f1": 0.0,
            "f1_macro": 0.0,
            "f1_yes": 0.0,
            "f1_no": 0.0,
            "majority_baseline": 0.0,
            "validity": 0.0,
            "label_distribution": "",
            "baseline_f1_macro": 0.0,
            "baseline_f1_yes": 0.0,
            "baseline_f1_no": 0.0,
        }

    acc = sub["is_correct"].mean()
    dist = sub["gt_norm"].value_counts(dropna=False)
    dist_str = ", ".join([f"{k}:{v}" for k, v in dist.items()])
    majority_acc = int(dist.max()) / n

    pred_valid = sub["bool_model_answer"].fillna("").astype(str).str.strip().str.lower()
    if sub["gt_norm"].isin(["yes", "no"]).all():
        valid = pred_valid.isin(["yes", "no"])
    else:
        scope = set(sub["gt_norm"].dropna().unique())
        valid = pred_valid.isin(scope) if scope else (pred_valid != "")
    validity = valid.mean()

    y_true = (sub["gt_norm"] == "yes").astype(int).to_numpy()
    y_pred = (sub["pred_label"] == "yes").astype(int).to_numpy()
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_yes = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_no = f1_score(y_true, y_pred, pos_label=0, zero_division=0)
    f1 = f1_yes

    maj_label = dist.idxmax()
    y_base = np.ones_like(y_true) if maj_label == "yes" else np.zeros_like(y_true)
    f1_macro_base = f1_score(y_true, y_base, average="macro", zero_division=0)
    f1_yes_base = f1_score(y_true, y_base, pos_label=1, zero_division=0)
    f1_no_base = f1_score(y_true, y_base, pos_label=0, zero_division=0)

    return {
        "n": n,
        "accuracy": acc,
        "f1": f1,
        "f1_macro": f1_macro,
        "f1_yes": f1_yes,
        "f1_no": f1_no,
        "majority_baseline": majority_acc,
        "validity": validity,
        "label_distribution": dist_str,
        "baseline_f1_macro": f1_macro_base,
        "baseline_f1_yes": f1_yes_base,
        "baseline_f1_no": f1_no_base,
    }

df = _ensure_columns(df)

GPT_METRICS = {
    "Ambiguity=yes": _compute_metrics(df[df["_amb_norm"] == "yes"]),
    "Ambiguity=no": _compute_metrics(df[df["_amb_norm"] == "no"]),
    "RuleType=Parent Rule": _compute_metrics(df[df["_rule_type"] == "Parent Rule"]),
    "RuleType=Atomic Rule": _compute_metrics(df[df["_rule_type"] == "Atomic Rule"]),
    "Overall": _compute_metrics(df),
}

# Optional unified dictionary for the final table
if "MODEL_METRICS" not in globals() or not isinstance(MODEL_METRICS, dict):
    MODEL_METRICS = {}
MODEL_METRICS["GPT-5.2"] = GPT_METRICS

GPT_METRICS

{'Ambiguity=yes': {'n': 81,
  'accuracy': np.float64(0.41975308641975306),
  'f1': 0.12244897959183673,
  'f1_macro': 0.37095900307025464,
  'f1_yes': 0.12244897959183673,
  'f1_no': 0.6194690265486725,
  'majority_baseline': 0.5679012345679012,
  'validity': np.float64(1.0),
  'label_distribution': 'yes:46, no:35',
  'baseline_f1_macro': 0.36220472440944884,
  'baseline_f1_yes': 0.7244094488188977,
  'baseline_f1_no': 0.0},
 'Ambiguity=no': {'n': 82,
  'accuracy': np.float64(0.5365853658536586),
  'f1': 0.27450980392156865,
  'f1_macro': 0.4735380878014923,
  'f1_yes': 0.27450980392156865,
  'f1_no': 0.672566371681416,
  'majority_baseline': 0.5365853658536586,
  'validity': np.float64(1.0),
  'label_distribution': 'yes:44, no:38',
  'baseline_f1_macro': 0.3492063492063492,
  'baseline_f1_yes': 0.6984126984126984,
  'baseline_f1_no': 0.0},
 'RuleType=Parent Rule': {'n': 163,
  'accuracy': np.float64(0.4785276073619632),
  'f1': 0.2,
  'f1_macro': 0.42300884955752216,
  'f1_yes': 0.2,


## 5. Final Summary Export

In [71]:
import pandas as pd

OUTPUT_SUMMARY_CSV = "3_1_yes_no_summary.csv"

SECTIONS = [
    "Ambiguity=yes",
    "Ambiguity=no",
    "Overall",
    "RuleType=Parent Rule",
    "RuleType=Atomic Rule",
]

METRIC_ORDER = [
    "n",
    "accuracy",
    "f1",
    "f1_macro",
    "f1_yes",
    "f1_no",
    "majority_baseline",
    "validity",
    "label_distribution",
    "baseline_f1_macro",
    "baseline_f1_yes",
    "baseline_f1_no",
]

# Expect precomputed metrics dicts from earlier cells.
# Supported shapes:
# 1) MODEL_METRICS = {"Claude Opus": {section: {metric: value}}, ...}
# 2) CLAUDE_METRICS / GEMINI_METRICS / GPT_METRICS with the same inner structure.

def get_model_metrics(model_name: str):
    if "MODEL_METRICS" in globals() and isinstance(MODEL_METRICS, dict):
        if model_name in MODEL_METRICS:
            return MODEL_METRICS[model_name]

    key_map = {
        "Claude Opus": ["CLAUDE_METRICS", "claude_metrics"],
        "Gemini 3 Pro Preview": ["GEMINI_METRICS", "gemini_metrics"],
        "GPT-5.2": ["GPT_METRICS", "gpt_metrics"],
    }
    for key in key_map.get(model_name, []):
        if key in globals() and isinstance(globals()[key], dict):
            return globals()[key]
    return None

MODEL_NAMES = ["Claude Opus", "Gemini 3 Pro Preview", "GPT-5.2"]

model_metrics = {name: get_model_metrics(name) for name in MODEL_NAMES}

rows = []
for section in SECTIONS:
    # dataset characteristics from the first available model
    dataset_desc = ""
    for name in MODEL_NAMES:
        mm = model_metrics.get(name)
        if mm and section in mm:
            n = mm[section].get("n", "")
            labels = mm[section].get("label_distribution", "")
            dataset_desc = f"n={n}, labels={labels}" if n != "" or labels != "" else ""
            break

    for metric in METRIC_ORDER:
        row = {
            "Section": section,
            "Metric": metric,
            "Claude Opus": "",
            "Gemini 3 Pro Preview": "",
            "GPT-5.2": "",
            "Dataset characteristics": dataset_desc,
        }
        for name in MODEL_NAMES:
            mm = model_metrics.get(name)
            if mm and section in mm:
                row[name] = mm[section].get(metric, "")
        rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False)
print("Saved:", OUTPUT_SUMMARY_CSV)

OUTPUT_AMBIGUITY_CSV = "3_1_yes_no_summary_ambiguity_overall.csv"
AMBIGUITY_SECTIONS = ["Ambiguity=yes", "Ambiguity=no", "Overall"]
ambiguity_overall_df = summary_df[summary_df["Section"].isin(AMBIGUITY_SECTIONS)].copy()
ambiguity_overall_df["Section"] = pd.Categorical(
    ambiguity_overall_df["Section"],
    categories=AMBIGUITY_SECTIONS,
    ordered=True,
)
ambiguity_overall_df["Metric"] = pd.Categorical(
    ambiguity_overall_df["Metric"],
    categories=METRIC_ORDER,
    ordered=True,
)
ambiguity_overall_df = ambiguity_overall_df.sort_values(["Section", "Metric"]).reset_index(drop=True)
ambiguity_overall_df.to_csv(OUTPUT_AMBIGUITY_CSV, index=False)
print("Saved:", OUTPUT_AMBIGUITY_CSV)
ambiguity_overall_df


Saved: 3_1_yes_no_summary.csv
Saved: 3_1_yes_no_summary_ambiguity_overall.csv


,Section,Metric,Claude Opus,Gemini 3 Pro Preview,GPT-5.2,Dataset characteristics
0,Ambiguity=yes,n,81,81,81,"n=81, labels=yes:46, no:35"
1,Ambiguity=yes,accuracy,0.45679,0.432099,0.419753,"n=81, labels=yes:46, no:35"
2,Ambiguity=yes,f1,0.385965,0.264151,0.122449,"n=81, labels=yes:46, no:35"
3,Ambiguity=yes,f1_macro,0.526316,0.453176,0.370959,"n=81, labels=yes:46, no:35"
4,Ambiguity=yes,f1_yes,0.385965,0.264151,0.122449,"n=81, labels=yes:46, no:35"
5,Ambiguity=yes,f1_no,0.666667,0.642202,0.619469,"n=81, labels=yes:46, no:35"
6,Ambiguity=yes,majority_baseline,0.567901,0.567901,0.567901,"n=81, labels=yes:46, no:35"
7,Ambiguity=yes,validity,1.0,0.987654,1.0,"n=81, labels=yes:46, no:35"
8,Ambiguity=yes,label_distribution,"yes:46, no:35","yes:46, no:35","yes:46, no:35","n=81, labels=yes:46, no:35"
9,Ambiguity=yes,baseline_f1_macro,0.362205,0.362205,0.362205,"n=81, labels=yes:46, no:35"


### 5.1 Export Excel Report

Converts `3_1_yes_no_summary.csv` into a formatted `3_1_yes_no_summary.xlsx` report.

In [72]:
import pandas as pd
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
from openpyxl.utils import get_column_letter

# =========================
# Paths
# =========================
csv_path = Path("3_1_yes_no_summary.csv")
xlsx_path = csv_path.with_suffix(".xlsx")

# =========================
# Read source CSV
# =========================
df = pd.read_csv(csv_path)

# Expected column order in the CSV
required_columns = [
    "Section",
    "Metric",
    "Claude Opus",
    "Gemini 3 Pro Preview",
    "GPT-5.2",
    "Dataset characteristics",
]

missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

df = df[required_columns].copy()

# Preserve section order exactly as it appears in the CSV
section_order = list(dict.fromkeys(df["Section"].astype(str).tolist()))

# =========================
# Create workbook
# =========================
wb = Workbook()
ws = wb.active
ws.title = "YesNo Summary"

# =========================
# Styles
# =========================
thin = Side(style="thin", color="000000")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

section_fill = PatternFill("solid", fgColor="D9EAF7")
header_fill = PatternFill("solid", fgColor="EDEDED")

section_font = Font(bold=True, size=12)
header_font = Font(bold=True)
normal_font = Font(bold=False)

center = Alignment(horizontal="center", vertical="center")
center_wrap = Alignment(horizontal="center", vertical="center", wrap_text=True)
left_wrap = Alignment(horizontal="left", vertical="center", wrap_text=True)

# Final visible Excel columns (without Section column)
excel_headers = [
    "Metric",
    "Claude Opus",
    "Gemini 3 Pro Preview",
    "GPT-5.2",
    "Dataset characteristics",
]

# =========================
# Write content section by section
# =========================
current_row = 1

for section in section_order:
    sub = df[df["Section"].astype(str) == str(section)].copy()

    # 1) Merged section header across all 5 visible columns
    ws.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=5)
    section_cell = ws.cell(row=current_row, column=1, value=section)
    section_cell.font = section_font
    section_cell.alignment = center
    section_cell.fill = section_fill
    section_cell.border = border

    # Apply border/fill to merged range cells too
    for col in range(1, 6):
        cell = ws.cell(row=current_row, column=col)
        cell.fill = section_fill
        cell.border = border

    current_row += 1

    # 2) Column header row
    for col_idx, header in enumerate(excel_headers, start=1):
        cell = ws.cell(row=current_row, column=col_idx, value=header)
        cell.font = header_font
        cell.alignment = center_wrap
        cell.fill = header_fill
        cell.border = border

    current_row += 1

    # 3) Data rows
    for _, row in sub.iterrows():
        values = [
            row["Metric"],
            row["Claude Opus"],
            row["Gemini 3 Pro Preview"],
            row["GPT-5.2"],
            row["Dataset characteristics"],
        ]

        for col_idx, value in enumerate(values, start=1):
            cell = ws.cell(row=current_row, column=col_idx, value=value)

            # Alignment by column
            if col_idx in [1, 5]:
                cell.alignment = left_wrap
            else:
                cell.alignment = center_wrap

            cell.font = normal_font
            cell.border = border

        current_row += 1

    # Empty spacer row between sections
    current_row += 1

# =========================
# Column widths
# =========================
column_widths = {
    1: 22,  # Metric
    2: 16,  # Claude Opus
    3: 24,  # Gemini 3 Pro Preview
    4: 14,  # GPT-5.2
    5: 34,  # Dataset characteristics
}

for col_idx, width in column_widths.items():
    ws.column_dimensions[get_column_letter(col_idx)].width = width

# Optional: set a comfortable default row height for wrapped text
for row in range(1, ws.max_row + 1):
    ws.row_dimensions[row].height = 20

# Freeze top row below first section is not very useful here,
# so instead keep the sheet simple without freezing panes.
# If you want, you can freeze at a specific point later.

# =========================
# Save workbook
# =========================
wb.save(xlsx_path)

print(f"Excel file created: {xlsx_path}")

Excel file created: 3_1_yes_no_summary.xlsx
